# Modelado de datos en SQL — Emisiones GEI Argentina (1990-2022)

Este notebook continúa el trabajo de limpieza y exploración realizado en `01_exploracion_limpieza.ipynb`. Acá tomamos el dataset ya limpio y lo llevamos a una base de datos relacional (SQLite), organizada en un **modelo en estrella**, y construimos sobre ella las vistas SQL que después se importan en Power BI para armar el dashboard final.

## Índice

1. Preparación de los datos y conexión a la base
2. Diseño del modelo: esquema en estrella
3. Carga y verificación de las tablas
4. Vistas analíticas (una por cada pregunta del proyecto)
5. Exportación a CSV para Power BI

In [26]:
import sqlite3
import pandas as pd

## 1. Preparación de los datos y conexión a la base

Partimos del CSV ya limpio, exportado al final del notebook anterior. Renombramos dos columnas (`año` → `anio`, `valor_en_toneladas_de_co2e` → `valor_tco2e`) para que sean más cómodas de escribir en las consultas SQL, sin acentos ni nombres largos.

Usamos **SQLite** como motor de base de datos: no necesita instalar ni levantar un servidor, guarda todo en un único archivo (`proyecto_gei.db`) y viene incluido en Python (`sqlite3`), lo que lo hace ideal para un proyecto de portfolio que después se sube tal cual a GitHub.

In [27]:
df = pd.read_csv("E:\\PROYECTOS PYTHON\\proyecto-GEI\\data\\processed\\emisiones_datos_totales_1990_2022_coma.csv")
df = df.rename(columns={
    "año": "anio",
    "valor_en_toneladas_de_co2e": "valor_tco2e",
})
df.shape

(7737, 9)

In [28]:
conn = sqlite3.connect("E:\\PROYECTOS PYTHON\\proyecto-GEI\\sql\\proyecto_gei.db")
cur = conn.cursor()

## 2. Diseño del modelo: esquema en estrella

En vez de guardar todo en una sola tabla plana (como el CSV original), organizamos los datos en un **esquema en estrella**, el modelo más usado en BI porque es el que mejor entienden herramientas como Power BI:

- **`fact_emisiones`** (tabla de hechos): una fila por cada medición — año, sector, gas y la cantidad de toneladas de CO2 equivalente. Es la tabla grande, con toda la información numérica.
- **`dim_sector`** y **`dim_gas`** (tablas de dimensión): tablas chicas, tipo "catálogo", con la lista de sectores y de gases. En vez de repetir el texto "Agricultura, Ganadería..." miles de veces dentro de `fact_emisiones`, guardamos un número (`sector_id`) que apunta a esa tabla.

Esto evita duplicar texto (ahorra espacio), evita errores de tipeo entre filas, y es exactamente la estructura que después Power BI va a usar para armar las relaciones del modelo.

In [29]:
cur.executescript("""
DROP TABLE IF EXISTS fact_emisiones;
DROP TABLE IF EXISTS dim_sector;
DROP TABLE IF EXISTS dim_gas;

CREATE TABLE dim_sector (
    sector_id   INTEGER PRIMARY KEY AUTOINCREMENT,
    sector      TEXT NOT NULL UNIQUE
);

CREATE TABLE dim_gas (
    gas_id      INTEGER PRIMARY KEY AUTOINCREMENT,
    tipo_de_gas TEXT NOT NULL UNIQUE,
    nombre_gas  TEXT NOT NULL
);

CREATE TABLE fact_emisiones (
    id                  INTEGER PRIMARY KEY AUTOINCREMENT,
    anio                INTEGER NOT NULL,
    sector_id           INTEGER NOT NULL REFERENCES dim_sector(sector_id),
    actividad           TEXT NOT NULL,
    subactividad        TEXT NOT NULL,
    categoria           TEXT NOT NULL,
    id_ipcc             TEXT NOT NULL,
    gas_id              INTEGER NOT NULL REFERENCES dim_gas(gas_id),
    valor_tco2e         REAL NOT NULL,
    tipo_de_registro    TEXT NOT NULL
);
""")
conn.commit()

## 3. Carga y verificación de las tablas

Con el esquema ya creado, cargamos primero las dos dimensiones (`dim_sector`, `dim_gas`) y después la tabla de hechos. Después de cada carga consultamos la tabla para confirmar que los datos entraron como se esperaba, y al final comparamos el total general contra el número ya validado en el notebook de exploración (12398.33 tCO2e) para asegurarnos de que no se perdió ni se duplicó ningún registro en el camino.

In [30]:
sectores = sorted(df["sector"].unique())
cur.executemany("INSERT INTO dim_sector (sector) VALUES (?)", [(s,) for s in sectores])

nombres_gas = {
    "CO2": "Dióxido de carbono",
    "CH4": "Metano",
    "N2O": "Óxido nitroso",
    "HFC": "Hidrofluorocarbonos",
    "PFC": "Perfluorocarbonos",
}
gases = sorted(df["tipo_de_gas"].unique())
cur.executemany(
    "INSERT INTO dim_gas (tipo_de_gas, nombre_gas) VALUES (?, ?)",
    [(g, nombres_gas.get(g, g)) for g in gases],
)
conn.commit()

pd.read_sql("SELECT * FROM dim_sector", conn)

,sector_id,sector
0,1,"Agricultura, Ganadería, Silvicultura y Otros U..."
1,2,Energía
2,3,Procesos industriales y uso de productos
3,4,Residuos


Con las dimensiones ya cargadas, cada sector y cada gas tiene su propio ID autogenerado (`sector_id`, `gas_id`). El siguiente paso es traer esos IDs y usarlos para reemplazar el texto de sector y gas en el dataset original, antes de cargar la tabla de hechos — así es como se arma la relación entre la tabla de hechos y las dimensiones.

In [31]:
sector_ids = pd.read_sql("SELECT sector_id, sector FROM dim_sector", conn)
gas_ids = pd.read_sql("SELECT gas_id, tipo_de_gas FROM dim_gas", conn)

df = df.merge(sector_ids, on="sector", how="left")
df = df.merge(gas_ids, on="tipo_de_gas", how="left")

assert df["sector_id"].isnull().sum() == 0
assert df["gas_id"].isnull().sum() == 0

columnas_fact = ["anio", "sector_id", "actividad", "subactividad",
                  "categoria", "id_ipcc", "gas_id", "valor_tco2e", "tipo_de_registro"]

df[columnas_fact].to_sql("fact_emisiones", conn, if_exists="append", index=False)
conn.commit()

In [32]:
total_filas = cur.execute("SELECT COUNT(*) FROM fact_emisiones").fetchone()[0]
total_valor = cur.execute("SELECT SUM(valor_tco2e) FROM fact_emisiones").fetchone()[0]
print(f'Filas: {total_filas}\nValor total TCO2e: {round(total_valor, 2)}')

Filas: 7737
Valor total TCO2e: 12398.33


## 4. Vistas analíticas

Una **vista** en SQL es una consulta guardada que se comporta como si fuera una tabla, sin duplicar los datos: cada vez que se consulta, SQLite vuelve a calcular el resultado a partir de las tablas originales. Esto permite dejar ya resuelto, del lado de la base de datos, todo cálculo que no depende de los filtros que elija después el usuario del dashboard.

Construimos 7 vistas, una por cada pregunta del proyecto. Son agregaciones fijas (totales, porcentajes, rankings) que se importan directamente a Power BI como tablas. Los cálculos que sí dependen de lo que el usuario filtre en el dashboard (por ejemplo, un total que cambie según el sector seleccionado) no van acá — esos se resuelven después con medidas DAX, directamente en Power BI.

### Balance anual (emisiones, remociones, neto)

Responde la pregunta principal del proyecto: ¿cómo evolucionó el balance de gases de efecto invernadero año a año? Separamos las emisiones (valores positivos) de las remociones (valores negativos, asociadas a bosques y tierras que capturan carbono) para poder mostrarlas por separado en el dashboard, además del balance neto total.

In [33]:
cur.executescript("""
DROP VIEW IF EXISTS vw_balance_anual;
CREATE VIEW vw_balance_anual AS
SELECT
    anio,
    ROUND(SUM(CASE WHEN valor_tco2e > 0 THEN valor_tco2e ELSE 0 END), 2) AS emisiones,
    ROUND(SUM(CASE WHEN valor_tco2e < 0 THEN valor_tco2e ELSE 0 END), 2) AS remociones,
    ROUND(SUM(valor_tco2e), 2) AS balance_neto
FROM fact_emisiones
GROUP BY anio
ORDER BY anio;
""")
conn.commit()

In [34]:
cur.execute("""
SELECT SUM(balance_neto) AS total_balance
FROM vw_balance_anual;
""")

resultado = cur.fetchone()
print("Suma total del balance_neto:", resultado[0])

Suma total del balance_neto: 12398.33


In [35]:
pd.read_sql("SELECT * FROM vw_balance_anual", conn)

,anio,emisiones,remociones,balance_neto
0,1990,313.43,-39.71,273.72
1,1991,343.90,-39.72,304.18
2,1992,346.61,-34.57,312.04
3,1993,353.19,-43.68,309.51
4,1994,352.98,-37.15,315.83
5,1995,363.95,-42.23,321.72
6,1996,395.43,-40.05,355.38
7,1997,405.94,-53.57,352.37
8,1998,411.57,-35.12,376.45
9,1999,409.13,-41.11,368.02


### Totales y % de participación por sector

Responde qué sectores explican la mayor parte de las emisiones acumuladas entre 1990 y 2022. Además del total en toneladas de CO2 equivalente, calculamos el porcentaje que representa cada sector sobre el total general, listo para un gráfico de torta o de barras en Power BI.

In [36]:
cur.executescript("""
DROP VIEW IF EXISTS vw_totales_sector;
CREATE VIEW vw_totales_sector AS
SELECT
    s.sector,
    ROUND(SUM(f.valor_tco2e), 2) AS total_tco2e,
    ROUND(100.0 * SUM(f.valor_tco2e) / (SELECT SUM(valor_tco2e) FROM fact_emisiones), 1) AS participacion_pct
FROM fact_emisiones f
JOIN dim_sector s ON f.sector_id = s.sector_id
GROUP BY s.sector
ORDER BY total_tco2e DESC;""")
conn.commit()

In [37]:
pd.read_sql("SELECT * FROM vw_totales_sector", conn)

,sector,total_tco2e,participacion_pct
0,"Agricultura, Ganadería, Silvicultura y Otros U...",5899.92,47.6
1,Energía,5422.41,43.7
2,Residuos,576.18,4.6
3,Procesos industriales y uso de productos,499.82,4.0


### Evolución por año y sector (formato largo, para el gráfico de líneas)

Es la misma información que el balance anual, pero desagregada por sector y en "formato largo" (una fila por combinación de año y sector, en vez de una columna por sector). Este es el formato que necesita Power BI para armar un gráfico de líneas con una serie por sector.

In [38]:
cur.executescript("""
DROP VIEW IF EXISTS vw_evolucion_sector_anio;
CREATE VIEW vw_evolucion_sector_anio AS
SELECT
    f.anio,
    s.sector,
    ROUND(SUM(f.valor_tco2e), 2) AS total_tco2e
FROM fact_emisiones f
JOIN dim_sector s ON f.sector_id = s.sector_id
GROUP BY f.anio, s.sector
ORDER BY f.anio, s.sector;""")
conn.commit()

In [39]:
pd.read_sql("SELECT * FROM vw_evolucion_sector_anio", conn)

,anio,sector,total_tco2e
0,1990,"Agricultura, Ganadería, Silvicultura y Otros U...",151.13
1,1990,Energía,102.30
2,1990,Procesos industriales y uso de productos,8.48
3,1990,Residuos,11.81
4,1991,"Agricultura, Ganadería, Silvicultura y Otros U...",176.17
...,...,...,...
127,2021,Residuos,22.97
128,2022,"Agricultura, Ganadería, Silvicultura y Otros U...",153.55
129,2022,Energía,200.32
130,2022,Procesos industriales y uso de productos,23.59


### Variación interanual por sector (función de ventana: LAG)

Calcula cuánto cambió el total de cada sector respecto al año anterior, en valor absoluto y en porcentaje. Para eso usamos `LAG()`, una **función de ventana** que le permite a SQL "mirar" la fila del año anterior del mismo sector sin necesidad de un JOIN adicional. El primer año de cada sector no tiene un año anterior con el cual compararse, por eso su variación da nula (`NULL`).

In [40]:
cur.executescript("""
DROP VIEW IF EXISTS vw_variacion_sector;
CREATE VIEW vw_variacion_sector AS
WITH totales AS (
    SELECT f.anio, s.sector, SUM(f.valor_tco2e) AS total_tco2e
    FROM fact_emisiones f
    JOIN dim_sector s ON f.sector_id = s.sector_id
    GROUP BY f.anio, s.sector
)
SELECT
    anio,
    sector,
    ROUND(total_tco2e, 2) AS total_tco2e,
    ROUND(total_tco2e - LAG(total_tco2e) OVER (PARTITION BY sector ORDER BY anio), 2) AS variacion_absoluta,
    ROUND(
        100.0 * (total_tco2e - LAG(total_tco2e) OVER (PARTITION BY sector ORDER BY anio))
        / NULLIF(ABS(LAG(total_tco2e) OVER (PARTITION BY sector ORDER BY anio)), 0)
    , 1) AS variacion_pct
FROM totales
ORDER BY sector, anio;""")
conn.commit()

In [41]:
pd.read_sql("SELECT * FROM vw_variacion_sector WHERE anio = 2022", conn)

,anio,sector,total_tco2e,variacion_absoluta,variacion_pct
0,2022,"Agricultura, Ganadería, Silvicultura y Otros U...",153.55,-6.74,-4.2
1,2022,Energía,200.32,7.49,3.9
2,2022,Procesos industriales y uso de productos,23.59,1.48,6.7
3,2022,Residuos,23.33,0.36,1.6


In [42]:
pd.read_sql("SELECT * FROM vw_variacion_sector", conn)

,anio,sector,total_tco2e,variacion_absoluta,variacion_pct
0,1990,"Agricultura, Ganadería, Silvicultura y Otros U...",151.13,NaN,NaN
1,1991,"Agricultura, Ganadería, Silvicultura y Otros U...",176.17,25.04,16.6
2,1992,"Agricultura, Ganadería, Silvicultura y Otros U...",180.24,4.07,2.3
3,1993,"Agricultura, Ganadería, Silvicultura y Otros U...",174.97,-5.27,-2.9
4,1994,"Agricultura, Ganadería, Silvicultura y Otros U...",174.35,-0.62,-0.4
...,...,...,...,...,...
127,2018,Residuos,22.03,0.43,2.0
128,2019,Residuos,22.58,0.55,2.5
129,2020,Residuos,22.63,0.05,0.2
130,2021,Residuos,22.97,0.34,1.5


### Composición % por tipo de gas y año (función de ventana: SUM() OVER)

Responde qué gas explica la mayor parte de las emisiones en cada año, y cómo cambió esa composición con el tiempo. Usamos `SUM() OVER`, otra función de ventana, que calcula el total del año sin colapsar las filas como haría un `GROUP BY` tradicional — así podemos dividir cada gas por el total de su propio año dentro de la misma consulta.

In [43]:
cur.executescript("""
DROP VIEW IF EXISTS vw_composicion_gas_anio;
CREATE VIEW vw_composicion_gas_anio AS
WITH totales_gas AS (
    SELECT f.anio, g.tipo_de_gas, SUM(f.valor_tco2e) AS total_tco2e
    FROM fact_emisiones f
    JOIN dim_gas g ON f.gas_id = g.gas_id
    GROUP BY f.anio, g.tipo_de_gas
)
SELECT
    anio,
    tipo_de_gas,
    ROUND(total_tco2e, 2) AS total_tco2e,
    ROUND(100.0 * total_tco2e / SUM(total_tco2e) OVER (PARTITION BY anio), 1) AS participacion_pct
FROM totales_gas
ORDER BY anio, tipo_de_gas;""")
conn.commit()

In [44]:
pd.read_sql("SELECT * FROM vw_composicion_gas_anio", conn)

,anio,tipo_de_gas,total_tco2e,participacion_pct
0,1990,CH4,115.93,42.4
1,1990,CO2,141.67,51.8
2,1990,N2O,16.07,5.9
3,1990,PFC,0.05,0.0
4,1991,CH4,118.19,38.9
...,...,...,...,...
153,2022,CH4,129.35,32.3
154,2022,CO2,239.44,59.7
155,2022,HFC,6.56,1.6
156,2022,N2O,25.43,6.3


### Top 10 categorías 2018-2022 (función de ventana: RANK)

Identifica las 10 categorías de actividad individuales (por ejemplo "Bovinos de Carne" o "Transporte terrestre por carretera") que más contribuyeron al balance en los últimos 5 años del dataset (2018-2022). Usamos `RANK()` para ordenar las categorías de mayor a menor y quedarnos solo con las primeras 10.

In [45]:
cur.executescript("""
DROP VIEW IF EXISTS vw_top_categorias_recientes;
CREATE VIEW vw_top_categorias_recientes AS
WITH totales_categoria AS (
    SELECT categoria, SUM(valor_tco2e) AS total_tco2e
    FROM fact_emisiones
    WHERE anio >= 2018
    GROUP BY categoria
),
rankeadas AS (
    SELECT categoria, total_tco2e, RANK() OVER (ORDER BY total_tco2e DESC) AS ranking
    FROM totales_categoria
)
SELECT ranking, categoria, ROUND(total_tco2e, 2) AS total_tco2e
FROM rankeadas
WHERE ranking <= 10
ORDER BY ranking;""")
conn.commit()

In [46]:
pd.read_sql("SELECT * FROM vw_top_categorias_recientes", conn)

,ranking,categoria,total_tco2e
0,1,Bovinos de Carne,361.49
1,2,Transporte terrestre por carretera,225.97
2,3,Producción pública de electricidad y calor,196.20
3,4,Tierras forestales convertidas en pastizales,172.58
4,5,Residencial,121.01
5,6,Tierras forestales convertidas en Tierras de c...,109.02
6,7,Fugitivas producción de Gas Natural,100.72
7,8,Industria no especificada:,53.84
8,9,Fabricación de combustibles sólidos y otras in...,53.68
9,10,Bovinos Lecheros,45.32


### Actividades dentro de Energía y Agricultura/Ganadería

Muestra el detalle de actividades dentro de los dos sectores con mayor peso en el balance (Energía y Agricultura/Ganadería), para poder explicar en el dashboard qué actividades puntuales explican el total de cada uno de esos sectores.

In [47]:
cur.executescript("""
DROP VIEW IF EXISTS vw_actividades_sector_principal;
CREATE VIEW vw_actividades_sector_principal AS
SELECT
    s.sector,
    f.actividad,
    ROUND(SUM(f.valor_tco2e), 2) AS total_tco2e
FROM fact_emisiones f
JOIN dim_sector s ON f.sector_id = s.sector_id
WHERE s.sector IN ('Energía', 'Agricultura, Ganadería, Silvicultura y Otros Usos de la Tierra')
GROUP BY s.sector, f.actividad
ORDER BY s.sector, total_tco2e DESC;""")
conn.commit()

In [48]:
pd.read_sql("SELECT * FROM vw_actividades_sector_principal", conn)

,sector,actividad,total_tco2e
0,"Agricultura, Ganadería, Silvicultura y Otros U...",Ganadería,2701.09
1,"Agricultura, Ganadería, Silvicultura y Otros U...",Tierra,2368.58
2,"Agricultura, Ganadería, Silvicultura y Otros U...",Uso de Suelos,830.25
3,Energía,Actividades de quema de combustible,4669.41
4,Energía,Emisiones fugitivas provenientes de la fabrica...,753.00


## 5. Exportación a CSV para Power BI

Exportamos la tabla de hechos, las dos dimensiones y las 7 vistas a archivos CSV. Estos son los mismos archivos que después se importan manualmente en Power BI (Obtener datos > Texto/CSV), donde se arma el modelo de datos definitivo (las relaciones entre `fact_emisiones` y las dos dimensiones) y se construyen las medidas DAX para los cálculos que sí dependen de los filtros del dashboard.

In [49]:
tablas_y_vistas = [
    "fact_emisiones", "dim_sector", "dim_gas",
    "vw_balance_anual", "vw_totales_sector", "vw_evolucion_sector_anio",
    "vw_variacion_sector", "vw_composicion_gas_anio",
    "vw_top_categorias_recientes", "vw_actividades_sector_principal",
]

for nombre in tablas_y_vistas:
    pd.read_sql(f"SELECT * FROM {nombre}", conn).to_csv(
        f"../data/powerbi/{nombre}.csv", index=False, decimal=","
    )
    print(nombre, "exportado")

conn.close()

fact_emisiones exportado
dim_sector exportado
dim_gas exportado
vw_balance_anual exportado
vw_totales_sector exportado
vw_evolucion_sector_anio exportado
vw_variacion_sector exportado
vw_composicion_gas_anio exportado
vw_top_categorias_recientes exportado
vw_actividades_sector_principal exportado


## Conclusiones

- El dataset limpio se modeló en un esquema en estrella (`fact_emisiones` + `dim_sector` + `dim_gas`) sobre SQLite, sin necesidad de un servidor de base de datos.
- Se construyeron 7 vistas analíticas que resuelven las principales preguntas del proyecto, usando CTEs y funciones de ventana (`LAG`, `RANK`, `SUM() OVER`) en vez de subconsultas anidadas.
- Cada vista se verificó contra los totales ya validados en el notebook de exploración (12398.33 tCO2e), confirmando que el modelado no perdió ni duplicó información.
- **Próximo paso:** importar estas tablas y vistas a Power BI, armar las relaciones del modelo y construir el dashboard con las medidas DAX que dependen de los filtros del usuario.